# Join & Preprocessing Pipeline

## Background

This notebook integrates two independent data sources — ClinicalTrials.gov and OpenFDA — into a single model-ready dataset. ClinicalTrials.gov provides trial design metadata and outcome labels; OpenFDA contributes drug regulatory context (application type, approval year, therapeutic class, adverse-event signals) that is only available for drug and biologic interventions. Because the two sources share no common key, they are linked by fuzzy matching on drug names, a deliberate design choice that trades recall for precision and is documented in Section 3 below.

Trials with ambiguous outcome statuses (Suspended, Unknown) are excluded at this stage. Non-drug trials (devices, behavioral) are retained with OpenFDA features set to null and flagged via `has_fda_record = False`.

## Coverage

- Relational database (DuckDB) — SQL filtering and joining over raw CSVs
- Record linking / fuzzy matching — token-sort ratio ≥ 85 via rapidfuzz
- Data join — ClinicalTrials + OpenFDA on drug name
- Hypothesis testing — chi-squared (sponsor × outcome) and Mann-Whitney U (enrollment)
- Feature engineering — derived fields, log transforms, eligibility age parsing
- Output — `joined_dataset.parquet` (53,628 rows × 40 columns)

In [ ]:
# !pip install duckdb polars rapidfuzz scipy

In [1]:
import duckdb
import polars as pl
from rapidfuzz import process, fuzz
from scipy import stats
import re
import warnings
warnings.filterwarnings('ignore')

CT_PATH  = 'clinical_trials_raw.csv'
FDA_PATH = 'openfda_processed.csv'
OUT_PATH = 'joined_dataset.parquet'

print('Libraries loaded.')

Libraries loaded.


## 1. Load Raw Data into DuckDB
Both CSVs are registered as views so we can query them with SQL.

In [2]:
con = duckdb.connect()  # in-memory DB

con.execute(f"CREATE VIEW ct  AS SELECT * FROM read_csv_auto('{CT_PATH}',  header=true, null_padding=true)")
con.execute(f"CREATE VIEW fda AS SELECT * FROM read_csv_auto('{FDA_PATH}', header=true, null_padding=true)")

print(con.execute('SELECT COUNT(*) FROM ct').fetchone()[0],  'CT rows')
print(con.execute('SELECT COUNT(*) FROM fda').fetchone()[0], 'FDA rows')

114022 CT rows
12344 FDA rows


## 2. Filter to Drug Trials with a Valid Target Label
Keep only `COMPLETED`, `TERMINATED`, `WITHDRAWN` trials that are marked as drug trials.
Ambiguous statuses (`SUSPENDED`, `UNKNOWN`) are excluded and documented here.

In [3]:
# Distribution of all statuses before filtering
status_dist = con.execute("""
    SELECT overall_status, COUNT(*) AS n
    FROM ct
    GROUP BY overall_status
    ORDER BY n DESC
""").pl()
print(status_dist)

KEEP_STATUSES = ('COMPLETED', 'TERMINATED', 'WITHDRAWN')

drug_trials = con.execute("""
    SELECT *
    FROM ct
    WHERE is_drug_trial = true
      AND overall_status IN ('COMPLETED', 'TERMINATED', 'WITHDRAWN')
      AND drug_names IS NOT NULL
      AND drug_names != ''
""").pl()

print(f'\nDrug trials with valid label: {len(drug_trials):,}')
print(drug_trials['overall_status'].value_counts())

shape: (3, 2)
┌────────────────┬───────┐
│ overall_status ┆ n     │
│ ---            ┆ ---   │
│ str            ┆ i64   │
╞════════════════╪═══════╡
│ COMPLETED      ┆ 69541 │
│ TERMINATED     ┆ 32878 │
│ WITHDRAWN      ┆ 11603 │
└────────────────┴───────┘

Drug trials with valid label: 53,628
shape: (3, 2)
┌────────────────┬───────┐
│ overall_status ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ TERMINATED     ┆ 19694 │
│ COMPLETED      ┆ 28219 │
│ WITHDRAWN      ┆ 5715  │
└────────────────┴───────┘


## 3. Record Linking — Fuzzy Drug Name Matching

Drug names in ClinicalTrials (`drug_names` column, pipe-separated) and OpenFDA (`substance_name`, `generic_name`, `brand_name`) often differ in formatting. We:
1. Build a normalised lookup set from FDA names.
2. For each trial, explode the pipe-separated drug list and find the best fuzzy match.
3. Accept matches with `token_sort_ratio ≥ 85` (conservative threshold).
4. Map matched FDA names back to `application_number` for the SQL join.

In [4]:
def normalize(s: str) -> str:
    if not s:
        return ''
    s = s.lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[^a-z0-9 ]', '', s)
    return s.strip()

fda_df = con.execute("SELECT application_number, substance_name, generic_name, brand_name FROM fda").pl()

# Build flat lookup: norm_name -> application_number (first match wins)
fda_lookup: dict[str, str] = {}
for row in fda_df.iter_rows(named=True):
    for field in ('substance_name', 'generic_name', 'brand_name'):
        raw = row[field] or ''
        for part in raw.split(','):
            normed = normalize(part)
            if normed and normed not in fda_lookup:
                fda_lookup[normed] = row['application_number']

fda_choices = list(fda_lookup.keys())
print(f'FDA name variants for matching: {len(fda_choices):,}')

FDA name variants for matching: 6,046


In [5]:
FUZZY_THRESHOLD = 85

def match_drug(drug_name: str) -> tuple[str | None, str | None, float]:
    """Return (norm_fda_name, application_number, score) or (None, None, 0)."""
    normed = normalize(drug_name)
    if not normed:
        return None, None, 0.0
    # Exact hit first (fast path)
    if normed in fda_lookup:
        return normed, fda_lookup[normed], 100.0
    result = process.extractOne(normed, fda_choices, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= FUZZY_THRESHOLD:
        matched_name, score, _ = result
        return matched_name, fda_lookup[matched_name], float(score)
    return None, None, 0.0

# Explode pipe-separated drug_names, match each, keep best per trial
records = []
for row in drug_trials.iter_rows(named=True):
    best_app, best_score = None, 0.0
    for drug in (row['drug_names'] or '').split('|'):
        _, app_num, score = match_drug(drug)
        if score > best_score:
            best_app, best_score = app_num, score
    records.append({'nct_id': row['nct_id'], 'matched_app_number': best_app, 'match_score': best_score})

match_df = pl.DataFrame(records)
matched   = match_df.filter(pl.col('matched_app_number').is_not_null())
unmatched = match_df.filter(pl.col('matched_app_number').is_null())
print(f'Matched:   {len(matched):,}  ({100*len(matched)/len(match_df):.1f}%)')
print(f'Unmatched: {len(unmatched):,}  ({100*len(unmatched)/len(match_df):.1f}%)')
print(f'Score distribution (matched):'); print(matched['match_score'].describe())

Matched:   23,171  (43.2%)
Unmatched: 30,457  (56.8%)
Score distribution (matched):
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 23171.0   │
│ null_count ┆ 0.0       │
│ mean       ┆ 99.169344 │
│ std        ┆ 2.960508  │
│ min        ┆ 85.0      │
│ 25%        ┆ 100.0     │
│ 50%        ┆ 100.0     │
│ 75%        ┆ 100.0     │
│ max        ┆ 100.0     │
└────────────┴───────────┘


## 4. SQL Join in DuckDB
Register the fuzzy-match bridge table and perform the join entirely in SQL.

In [20]:
con.register('drug_trials_view', drug_trials)
con.register('match_bridge',     match_df)

joined = con.execute("""
    SELECT
        t.*,
        m.matched_app_number,
        m.match_score,
        f.application_type,
        f.sponsor_name          AS fda_sponsor_name,
        f.marketing_status,
        f.therapeutic_class_epc,
        f.therapeutic_class_cs,
        f.mechanism_of_action,
        f.regulatory_action_type,
        f.review_priority,
        f.approval_year,
        f.product_type,
        f.route,
        f.dosage_form,
        f.submission_type,
        -- flag whether FDA record was found
        (m.matched_app_number IS NOT NULL) AS has_fda_record
    FROM drug_trials_view t
    LEFT JOIN match_bridge m USING (nct_id)
    LEFT JOIN fda f ON f.application_number = m.matched_app_number
""").pl()

print(f'Joined dataset: {joined.shape}')
print(joined.select('overall_status').to_series().value_counts())

Joined dataset: (53628, 45)
shape: (3, 2)
┌────────────────┬───────┐
│ overall_status ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ TERMINATED     ┆ 19694 │
│ COMPLETED      ┆ 28219 │
│ WITHDRAWN      ┆ 5715  │
└────────────────┴───────┘


## 5. Feature Engineering

In [21]:
def parse_age_years(s):
    """Convert '18 Years' / '6 Months' / None to float years."""
    if s is None:
        return None
    s = str(s).strip().lower()
    m = re.match(r'([\d.]+)\s*(year|month|week|day)', s)
    if not m:
        return None
    val, unit = float(m.group(1)), m.group(2)
    return val / (12 if unit == 'month' else 52 if unit == 'week' else 365 if unit == 'day' else 1)

# Pass 1: create base features (no cross-column dependencies)
df = joined.with_columns([
    pl.col('overall_status')
      .map_elements(lambda s: {'COMPLETED': 0, 'TERMINATED': 1, 'WITHDRAWN': 2}.get(s, -1), return_dtype=pl.Int8)
      .alias('label'),
    pl.col('start_date').str.slice(0, 4).cast(pl.Int32, strict=False).alias('start_year'),
    pl.col('primary_completion_date').str.slice(0, 4).cast(pl.Int32, strict=False).alias('primary_completion_year'),
    pl.col('has_dmc').cast(pl.Boolean, strict=False),
    pl.col('healthy_volunteers').cast(pl.Boolean, strict=False),
    pl.col('is_fda_regulated_drug').cast(pl.Boolean, strict=False),
    # log1p avoids -inf when enrollment_count == 0
    (pl.col('enrollment_count').cast(pl.Float64, strict=False) + 1).log(base=10).alias('log_enrollment'),
    pl.col('minimum_age').map_elements(parse_age_years, return_dtype=pl.Float64).alias('min_age_years'),
    pl.col('maximum_age').map_elements(parse_age_years, return_dtype=pl.Float64).alias('max_age_years'),
    (pl.col('review_priority') == 'PRIORITY').cast(pl.Int8).alias('priority_review'),
    pl.col('conditions').map_elements(lambda s: len(s.split('|')) if s else 0, return_dtype=pl.Int32).alias('num_conditions'),
    pl.col('drug_names').map_elements(lambda s: len(s.split('|')) if s else 0, return_dtype=pl.Int32).alias('num_drugs'),
    pl.col('lead_sponsor_class').alias('sponsor_class'),
    # enrollment_type flag: 1 if actual count reported, 0 if anticipated
    (pl.col('enrollment_type') == 'ACTUAL').cast(pl.Int8).alias('enrollment_actual'),
    # trial duration in days (completion_date - start_date)
    (pl.col('completion_date').str.to_date(format='%Y-%m-%d', strict=False)
     - pl.col('start_date').str.to_date(format='%Y-%m-%d', strict=False))
    .dt.total_days().alias('trial_duration_days'),
])

# Pass 2: derived features that depend on columns created in pass 1
df = df.with_columns([
    (pl.col('max_age_years') - pl.col('min_age_years')).alias('age_range_years'),
    (pl.col('start_year') - pl.col('approval_year').cast(pl.Int32, strict=False)).alias('years_since_approval'),
])

print('Feature engineering done.')
print(df.select(['label','start_year','log_enrollment','num_conditions','num_drugs',
                 'has_fda_record','enrollment_actual','trial_duration_days']).describe())

Feature engineering done.
shape: (9, 9)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ statistic ┆ label    ┆ start_yea ┆ log_enrol ┆ … ┆ num_drugs ┆ has_fda_r ┆ enrollmen ┆ trial_dur │
│ ---       ┆ ---      ┆ r         ┆ lment     ┆   ┆ ---       ┆ ecord     ┆ t_actual  ┆ ation_day │
│ str       ┆ f64      ┆ ---       ┆ ---       ┆   ┆ f64       ┆ ---       ┆ ---       ┆ s         │
│           ┆          ┆ f64       ┆ f64       ┆   ┆           ┆ f64       ┆ f64       ┆ ---       │
│           ┆          ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64       │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count     ┆ 53628.0  ┆ 53628.0   ┆ 53087.0   ┆ … ┆ 53628.0   ┆ 53628.0   ┆ 51580.0   ┆ 23760.0   │
│ null_coun ┆ 0.0      ┆ 0.0       ┆ 541.0     ┆ … ┆ 0.0       ┆ 0.0       ┆ 2048.0    ┆ 29868.0   │
│ t         ┆          ┆           ┆           ┆   

## 6. Hypothesis Testing

**H₀**: Industry-sponsored trials complete at the same rate as NIH-sponsored trials.

Use a chi-squared test of independence on the 2×3 contingency table (sponsor class × outcome).

In [22]:
from scipy.stats import chi2_contingency, mannwhitneyu

# --- Test 1: Sponsor class vs outcome ---
sponsor_outcome = con.execute("""
    SELECT lead_sponsor_class, overall_status, COUNT(*) AS n
    FROM drug_trials_view
    WHERE lead_sponsor_class IN ('INDUSTRY', 'NIH', 'OTHER_GOV', 'OTHER')
    GROUP BY 1, 2
""").pl()

pivot = sponsor_outcome.pivot(values='n', index='lead_sponsor_class', on='overall_status', aggregate_function='sum').fill_null(0)
print('Contingency table (sponsor class × outcome):'); print(pivot)

contingency = pivot.select(['COMPLETED','TERMINATED','WITHDRAWN']).to_numpy()
chi2, p, dof, expected = chi2_contingency(contingency)
print(f'\nChi-squared: {chi2:.2f},  df={dof},  p={p:.2e}')
print('Interpretation:', 'Significant (reject H₀)' if p < 0.05 else 'Not significant')

Contingency table (sponsor class × outcome):
shape: (4, 4)
┌────────────────────┬───────────┬────────────┬───────────┐
│ lead_sponsor_class ┆ COMPLETED ┆ TERMINATED ┆ WITHDRAWN │
│ ---                ┆ ---       ┆ ---        ┆ ---       │
│ str                ┆ i64       ┆ i64        ┆ i64       │
╞════════════════════╪═══════════╪════════════╪═══════════╡
│ INDUSTRY           ┆ 13657     ┆ 8572       ┆ 1641      │
│ OTHER_GOV          ┆ 424       ┆ 145        ┆ 59        │
│ NIH                ┆ 933       ┆ 665        ┆ 112       │
│ OTHER              ┆ 12645     ┆ 9936       ┆ 3767      │
└────────────────────┴───────────┴────────────┴───────────┘

Chi-squared: 954.44,  df=6,  p=6.38e-203
Interpretation: Significant (reject H₀)


In [23]:
# --- Test 2: Enrollment count — Completed vs Terminated (Mann-Whitney U) ---
completed  = df.filter(pl.col('overall_status') == 'COMPLETED')['enrollment_count'].drop_nulls().to_list()
terminated = df.filter(pl.col('overall_status') == 'TERMINATED')['enrollment_count'].drop_nulls().to_list()

stat, p2 = mannwhitneyu(completed, terminated, alternative='two-sided')
print(f'Mann-Whitney U (enrollment: completed vs terminated): U={stat:.0f}, p={p2:.2e}')
print('Interpretation:', 'Significant difference in enrollment' if p2 < 0.05 else 'No significant difference')

Mann-Whitney U (enrollment: completed vs terminated): U=392851423, p=0.00e+00
Interpretation: Significant difference in enrollment


In [24]:
# --- Test 3: Phase vs completion rate ---
phase_outcome = con.execute("""
    SELECT phases, overall_status, COUNT(*) AS n
    FROM drug_trials_view
    WHERE phases IS NOT NULL
    GROUP BY 1, 2
""").pl()
print('Phase × outcome counts:'); print(phase_outcome)

Phase × outcome counts:
shape: (24, 3)
┌───────────────┬────────────────┬──────┐
│ phases        ┆ overall_status ┆ n    │
│ ---           ┆ ---            ┆ ---  │
│ str           ┆ str            ┆ i64  │
╞═══════════════╪════════════════╪══════╡
│ NA            ┆ COMPLETED      ┆ 2616 │
│ PHASE2|PHASE3 ┆ TERMINATED     ┆ 536  │
│ PHASE1        ┆ TERMINATED     ┆ 3560 │
│ EARLY_PHASE1  ┆ TERMINATED     ┆ 320  │
│ PHASE4        ┆ COMPLETED      ┆ 3999 │
│ …             ┆ …              ┆ …    │
│ PHASE1        ┆ COMPLETED      ┆ 6468 │
│ PHASE2|PHASE3 ┆ COMPLETED      ┆ 620  │
│ EARLY_PHASE1  ┆ COMPLETED      ┆ 413  │
│ NA            ┆ TERMINATED     ┆ 1238 │
│ PHASE3        ┆ COMPLETED      ┆ 4958 │
└───────────────┴────────────────┴──────┘


## 7. Final Dataset Assembly & Export

In [25]:
MODEL_FEATURES = [
    # Trial design
    'phases', 'study_type', 'intervention_model', 'primary_purpose', 'masking',
    'enrollment_count', 'log_enrollment', 'enrollment_actual',
    'num_primary_outcomes', 'num_secondary_outcomes',
    'num_conditions', 'num_drugs', 'num_sites',
    # Sponsor
    'sponsor_class', 'num_collaborators',
    # Eligibility
    'min_age_years', 'max_age_years', 'age_range_years', 'sex', 'healthy_volunteers',
    # Trial flags
    'has_dmc', 'is_fda_regulated_drug',
    # Timeline
    'start_year', 'trial_duration_days',
    # FDA features (may be null if no match)
    'application_type', 'marketing_status', 'therapeutic_class_epc',
    'mechanism_of_action', 'review_priority', 'approval_year',
    'years_since_approval', 'priority_review', 'product_type', 'route', 'dosage_form',
    'submission_type', 'has_fda_record',
    # Target
    'label', 'overall_status', 'nct_id',
]

final = df.select([c for c in MODEL_FEATURES if c in df.columns])
print(f'Final dataset: {final.shape}')
print(final['label'].value_counts())

final.write_parquet(OUT_PATH)
print(f'Saved to {OUT_PATH}')

Final dataset: (53628, 40)
shape: (3, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ i8    ┆ u32   │
╞═══════╪═══════╡
│ 2     ┆ 5715  │
│ 1     ┆ 19694 │
│ 0     ┆ 28219 │
└───────┴───────┘
Saved to joined_dataset.parquet


In [26]:
# Quick DuckDB query on the final parquet as a sanity check
summary = duckdb.query(f"""
    SELECT overall_status,
           COUNT(*) AS n,
           ROUND(AVG(enrollment_count), 1) AS avg_enrollment,
           ROUND(AVG(CASE WHEN has_fda_record THEN 1.0 ELSE 0 END)*100, 1) AS pct_fda_matched,
           ROUND(AVG(num_sites), 1) AS avg_sites
    FROM '{OUT_PATH}'
    GROUP BY 1
    ORDER BY 1
""").pl()
print(summary)

shape: (3, 5)
┌────────────────┬───────┬────────────────┬─────────────────┬───────────┐
│ overall_status ┆ n     ┆ avg_enrollment ┆ pct_fda_matched ┆ avg_sites │
│ ---            ┆ ---   ┆ ---            ┆ ---             ┆ ---       │
│ str            ┆ i64   ┆ f64            ┆ f64             ┆ f64       │
╞════════════════╪═══════╪════════════════╪═════════════════╪═══════════╡
│ COMPLETED      ┆ 28219 ┆ 3609.6         ┆ 41.6            ┆ 12.0      │
│ TERMINATED     ┆ 19694 ┆ 180.1          ┆ 45.4            ┆ 13.6      │
│ WITHDRAWN      ┆ 5715  ┆ 2.3            ┆ 43.7            ┆ 1.8       │
└────────────────┴───────┴────────────────┴─────────────────┴───────────┘
